In [1]:
using Pkg
Pkg.activate(".")
# using Prime

  Activating project at `/home/mDisk/Prime-2`


In [2]:
using Distributed
addprocs(60)
@everywhere using Prime

In [3]:
using HDF5,DelimitedFiles,StatsBase
#Calculate cell-volume factor β
f = h5open("dataset/real_data/Human_skin_all.loom", "r")
X = read(f["layers/spliced"])+read(f["layers/unspliced"])
total_rd=sum(X,dims=2)
β = total_rd./mean(total_rd)

rdm = Matrix(readdlm("dataset/real_data/human_skin_hvg_spliced.csv",',')[2:end,2:end]')
rdn = Matrix(readdlm("dataset/real_data/human_skin_hvg_unspliced.csv", ',')[2:end,2:end]')

#cell number and gene number
n,p = size(rdm)
#Define cluster number and select model
k = 8
sel = 4

4

In [4]:
cluster_labels, γ = Prime.prime_cluster(
    rdn, rdm, β, k;
    sel = sel,
    maxiter = 20,  
)

┌ Info: PRIME iter 1, obj = 16.800738049080497
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 2, obj = 16.680488187174088
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 3, obj = 16.443901472147346
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 4, obj = 15.623756713443694
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 5, obj = 14.424759316156223
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 6, obj = 12.968981533711329
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 7, obj = 11.534245337989878
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 8, obj = 10.324371391685332
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 9, obj = 9.491649786107779
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 10, obj = 8.963536915595103
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 11, obj = 8.6439799834

([2, 2, 5, 3, 4, 8, 5, 5, 4, 4  …  4, 7, 1, 1, 7, 7, 3, 1, 8, 6], [8.908680788783722e-31 1.0367187710295562 … 1.0493719663448757e-28 3.73297097573885e-26; 1.1297702844826406e-31 1.0367187710295565 … 3.0497780396411642e-28 5.299660175110141e-26; … ; 3.234647226604928e-38 1.5120347418459023e-47 … 7.914850775267689e-32 1.0367187710295562; 5.718975717027709e-41 5.428064310863045e-47 … 1.0327153219940866e-37 7.780475402080495e-38])

In [7]:
using Clustering
cell_type = readdlm("dataset/real_data/human_skin_subclass_labels.csv",',')[2:end,2]
vec_numeric = replace(cell_type, 
"Smooth Muscle"   => 1,
"Macrophages/DC"   => 2,
"Pericytes"   => 3,
"Suprabasal Keratinocytes"   => 4,
"Endothelial Cells"     => 5,
"Fibroblasts"     => 6,
"Melanocytes" => 7,
"Basal Keratinocytes" => 8
)

println("ARI for PRIME clustering:",randindex(Int.(cluster_labels),Int.(vec_numeric))[1])
println("NMI for PRIME clustering:",mutualinfo(Int.(cluster_labels),Int.(vec_numeric)))



ARI for PRIME clustering:0.7617759680211702
NMI for PRIME clustering:0.7995871555719687
